In [ ]:
# !pip install Faker
# !pip install Xgboost

In [27]:
import pandas as pd

df = pd.read_csv('../../data/processed/ljh_preprocessed.csv')

In [28]:
import joblib
import xgboost

best_model = joblib.load('../../models/xgb_churn_v1/xgb_churn_v1.joblib')

In [30]:
from faker import Faker
import pandas as pd
import random
import numpy as np

fake = Faker()

# 1. 설정

np.random.seed(42)

# 1. 모델이 요구하는 정확한 20개 컬럼 리스트 (순서가 생명입니다!)
required_cols = [
    'gender', 'age_group', 'fixed_bet_amount', 'live_bet_amount', 
    'fixed_bet_cnt', 'live_bet_cnt', 'total_bet_cnt', 
    'fixed_active_days', 'live_active_days', 'fixed_hit_days', 
    'total_hit_days', 'fixed_win_rate', 'live_win_rate', 
    'total_win_rate', 'fixed_avg_roi', 'live_avg_roi', 
    'total_avg_roi', 'days_since_reg', 'days_to_first_deposit', 'days_to_first_bet'
]

n_samples = 1000
rows = []

# 2. 데이터 생성 루프 (전부 float64로 생성)
for _ in range(n_samples):
    # 기초 수치
    f_act = float(random.randint(1, 30))
    l_act = float(random.randint(1, 30))
    f_hit = float(random.randint(0, int(f_act)))
    t_hit = f_hit + float(random.randint(0, int(l_act)))
    
    # 신규 추가된 '날짜 관련 일수(Days)' 데이터
    d_reg = float(random.randint(1, 365))  # 가입한 지 얼마나 됐는지 (일)
    d_dep = float(random.randint(0, int(min(d_reg, 30)))) # 가입 후 입금까지 걸린 시간
    d_bet = d_dep + float(random.randint(0, 5)) # 입금 후 베팅까지 걸린 시간

    row = {
        'gender': float(random.randint(0, 1)),
        'age_group': float(random.randint(1, 5)),
        'fixed_bet_amount': float(random.randint(10000, 1000000)),
        'live_bet_amount': float(random.randint(5000, 800000)),
        'fixed_bet_cnt': float(random.randint(1, 50)),
        'live_bet_cnt': float(random.randint(1, 40)),
        'total_bet_cnt': float(random.randint(2, 90)),
        'fixed_active_days': f_act,
        'live_active_days': l_act,
        'fixed_hit_days': f_hit,
        'total_hit_days': t_hit,
        'fixed_win_rate': float(round(f_hit / f_act, 3)),
        'live_win_rate': float(round((t_hit - f_hit) / max(l_act, 1), 3)),
        'total_win_rate': float(round(t_hit / max(f_act + l_act, 1), 3)),
        'fixed_avg_roi': float(round(random.uniform(-1, 1), 3)),
        'live_avg_roi': float(round(random.uniform(-1, 1), 3)),
        'total_avg_roi': float(round(random.uniform(-1, 1), 3)),
        'days_since_reg': d_reg,            # 모델이 원한 컬럼 1
        'days_to_first_deposit': d_dep,     # 모델이 원한 컬럼 2
        'days_to_first_bet': d_bet          # 모델이 원한 컬럼 3
    }
    rows.append(row)

# 3. 데이터프레임 생성 및 타입 고정
df_fake = pd.DataFrame(rows)[required_cols].astype('float64')

In [31]:
df_fake

,gender,age_group,fixed_bet_amount,live_bet_amount,fixed_bet_cnt,live_bet_cnt,total_bet_cnt,fixed_active_days,live_active_days,fixed_hit_days,total_hit_days,fixed_win_rate,live_win_rate,total_win_rate,fixed_avg_roi,live_avg_roi,total_avg_roi,days_since_reg,days_to_first_deposit,days_to_first_bet
0,1.0,1.0,53752.0,660430.0,19.0,1.0,19.0,11.0,12.0,3.0,9.0,0.273,0.500,0.391,-0.521,-0.586,0.948,193.0,23.0,26.0
1,0.0,4.0,726041.0,552394.0,21.0,14.0,18.0,24.0,9.0,4.0,8.0,0.167,0.444,0.242,0.931,0.743,0.252,232.0,16.0,21.0
2,0.0,3.0,976320.0,369372.0,49.0,40.0,49.0,8.0,1.0,1.0,2.0,0.125,1.000,0.222,-0.461,0.366,0.018,263.0,11.0,16.0
3,0.0,3.0,680802.0,584761.0,45.0,18.0,62.0,24.0,14.0,24.0,37.0,1.000,0.929,0.974,-0.975,-0.024,-0.048,357.0,1.0,4.0
4,0.0,2.0,405354.0,445690.0,33.0,23.0,24.0,27.0,18.0,14.0,28.0,0.519,0.778,0.622,0.384,0.407,0.406,57.0,21.0,25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1.0,5.0,541316.0,65441.0,10.0,23.0,13.0,6.0,28.0,1.0,21.0,0.167,0.714,0.618,0.858,-0.885,0.387,97.0,25.0,29.0
996,0.0,3.0,459407.0,375526.0,31.0,22.0,16.0,19.0,16.0,14.0,22.0,0.737,0.500,0.629,0.364,0.315,-0.630,124.0,28.0,31.0
997,1.0,4.0,420121.0,45784.0,6.0,14.0,19.0,5.0,30.0,4.0,8.0,0.800,0.133,0.229,-0.972,-0.388,0.186,313.0,15.0,15.0
998,1.0,1.0,924187.0,470304.0,37.0,33.0,73.0,22.0,5.0,15.0,15.0,0.682,0.000,0.556,0.492,0.091,-0.570,353.0,17.0,21.0


In [32]:
best_model.predict(df_fake)

array([1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0,
       1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,